# AI Meeting Notes Extractor\n\nA browser-based meeting analysis dashboard with offline rule-based extraction and Gemini-powered analysis.\n\n## How to use\n1. Run the code cell below.\n2. Paste meeting notes into the notes area.\n3. Choose **Offline** or **Gemini 3 Flash**.\n4. For Gemini mode, paste your Gemini API key into the password field in the app. The key is not stored in this notebook.\n5. Click **Analyze Meeting Notes** and export the report as TXT, CSV, or PDF.\n\nThe Gemini key is sent from your browser to Google's Gemini API. Do not share your key or commit it to GitHub.\n

In [ ]:

from IPython.display import display, HTML, Javascript
import json

html = r"""
<div id="amn-v2">
<style>
#amn-v2 {
  background:#0b1020;color:#e5e7eb;padding:25px;
  border-radius:18px;font-family:Inter,Arial,sans-serif;
  border:1px solid #29334b;
}
#amn-v2 * {box-sizing:border-box}
#amn-v2 .header {display:flex;justify-content:space-between;
align-items:center;flex-wrap:wrap;gap:15px}
#amn-v2 .brand {font-size:27px;font-weight:850;color:#a5b4fc}
#amn-v2 .sub {color:#94a3b8;font-size:12px;margin-top:5px}
#amn-v2 .pill {background:#12352e;color:#6ee7b7;
padding:9px 13px;border-radius:20px;font-size:11px}
#amn-v2 .panel {background:#141b2e;border:1px solid #29334b;
border-radius:13px;padding:19px;margin-top:18px}
#amn-v2 .section {font-size:13px;font-weight:800;margin-bottom:14px}
#amn-v2 textarea,#amn-v2 select {
 width:100%;background:#0b1222;color:white;
 border:1px solid #384664;border-radius:9px;padding:13px;
 font:14px/1.6 Arial;
}
#amn-v2 textarea {min-height:155px;resize:vertical}
#amn-v2 button {border:0;border-radius:8px;padding:11px 14px;
font-weight:700;cursor:pointer;font-size:12px}
#amn-v2 button:disabled {opacity:.5;cursor:wait}
#amn-v2 .primary {width:100%;margin-top:12px;
background:linear-gradient(100deg,#6366f1,#9333ea);color:white}
#amn-v2 .secondary {background:#283550;color:#e2e8f0;margin:4px}
#amn-v2 .stats {display:grid;grid-template-columns:repeat(4,minmax(0,1fr));
gap:10px;margin-top:15px}
#amn-v2 .stat {background:#0d1528;border:1px solid #29334b;
padding:14px;border-radius:10px}
#amn-v2 .num {font-size:25px;font-weight:800;color:#a5b4fc}
#amn-v2 .stat small {color:#94a3b8;font-size:11px}
#amn-v2 .muted {color:#94a3b8;font-size:12px}
#amn-v2 .table-wrap {overflow-x:auto}
#amn-v2 table {width:100%;border-collapse:collapse;font-size:12px;
min-width:550px}
#amn-v2 th {text-align:left;color:#a5b4fc;background:#0b1222}
#amn-v2 td,#amn-v2 th {padding:12px 10px;
border-bottom:1px solid #29334b}
#amn-v2 .tag {padding:5px 8px;border-radius:15px;
font-size:10px;white-space:nowrap}
#amn-v2 .high {background:#48232b;color:#fca5a5}
#amn-v2 .medium {background:#49391c;color:#fcd34d}
#amn-v2 .low {background:#163d35;color:#6ee7b7}
#amn-v2 .output {white-space:pre-wrap;line-height:1.7;
background:#0b1222;padding:14px;border-radius:9px;
color:#dbeafe;font-size:13px;overflow-wrap:anywhere}
#amn-v2 .history-item {padding:12px;background:#0b1222;
border:1px solid #29334b;border-radius:8px;margin-top:8px}
#amn-v2 .history-item button {float:right}
#amn-v2 .footer {text-align:center;color:#64748b;
font-size:11px;margin-top:20px}
@media(max-width:600px) {
 #amn-v2 {padding:13px}
 #amn-v2 .stats {grid-template-columns:repeat(2,minmax(0,1fr))}
}
@media print {
 body * {visibility:hidden!important}
 #amn-v2,#amn-v2 #amn-report,#amn-v2 #amn-report * {
 visibility:visible!important
 }
 #amn-v2 {position:absolute;left:0;top:0;width:100%;
 background:white!important;color:black!important}
 #amn-v2 .no-print {display:none!important}
 #amn-v2 .panel,#amn-v2 .stat {background:white!important;
 color:black!important;border:1px solid #ccc}
 #amn-v2 .output {background:white!important;color:black!important}
 #amn-v2 table {color:black}
}
</style>

<div class="header">
 <div>
  <div class="brand">✦ MEETING INTELLIGENCE</div>
  <div class="sub">VERSION 2.0 · SMART MEETING ANALYTICS</div>
 </div>
 <div class="pill">● AI ENGINE READY</div>
</div>

<div class="panel no-print">
 <div class="section">01 / MEETING NOTES</div>
 <textarea id="amn-notes" placeholder="Paste your meeting notes here..."></textarea>
 <div class="muted" style="margin-top:8px">
  Add discussions, tasks, people, decisions and deadlines.
 </div>
</div>

<div class="panel no-print">
 <div class="section">02 / PROCESSING ENGINE</div>
 <label for="amn-api-key" class="muted" style="display:block;margin:0 0 8px">
  Gemini API key (needed only for Gemini mode)
 </label>
 <input id="amn-api-key" type="password" autocomplete="off"
  placeholder="Paste your Gemini API key here"
  style="width:100%;background:#0b1222;color:white;border:1px solid #384664;border-radius:9px;padding:13px;margin-bottom:14px">
 <select id="amn-mode">
  <option value="offline">Offline — Local extraction</option>
  <option value="live">Gemini 3 Flash — AI analysis</option>
 </select>
 <button class="primary" id="amn-run">✦ ANALYZE MEETING NOTES</button>
 <div id="amn-status" class="muted" style="margin-top:12px">
  Ready to analyze.
 </div>
</div>

<div id="amn-report">
 <div class="panel">
  <div class="section">03 / MEETING ANALYTICS</div>
  <div class="stats">
   <div class="stat"><div class="num" id="amn-total">0</div><small>Total tasks</small></div>
   <div class="stat"><div class="num" id="amn-high">0</div><small>High priority</small></div>
   <div class="stat"><div class="num" id="amn-pending">0</div><small>Pending tasks</small></div>
   <div class="stat"><div class="num" id="amn-done">0</div><small>Completed tasks</small></div>
  </div>
 </div>

 <div class="panel">
  <div class="section">04 / ACTION ITEM TRACKER</div>
  <div class="table-wrap">
   <table>
    <thead><tr>
     <th>Task</th><th>Owner</th><th>Deadline</th>
     <th>Priority</th><th>Status</th>
    </tr></thead>
    <tbody id="amn-tasks">
     <tr><td colspan="5" class="muted">Your tasks will appear here.</td></tr>
    </tbody>
   </table>
  </div>
 </div>

 <div class="panel">
  <div class="section">05 / MEETING SUMMARY</div>
  <div id="amn-summary" class="output">No analysis generated yet.</div>
  <div class="section" style="margin-top:17px">IMPORTANT DECISIONS</div>
  <div id="amn-decisions" class="output">No decisions yet.</div>
 </div>
</div>

<div class="panel no-print">
 <div class="section">06 / EXPORT CENTER</div>
 <button class="secondary" id="amn-txt">Download TXT</button>
 <button class="secondary" id="amn-csv">Download CSV</button>
 <button class="secondary" id="amn-pdf">Print / Save PDF</button>
 <button class="secondary" id="amn-copy">Copy report</button>
</div>

<div class="panel no-print">
 <div class="section">07 / MEETING HISTORY</div>
 <div id="amn-history" class="muted">No previous analyses in this session.</div>
</div>

<div class="footer">MEETING INTELLIGENCE · AI PRODUCTIVITY SUITE</div>
</div>
"""

display(HTML(html))

js = r"""
(() => {
 const root = document.getElementById("amn-v2");
 if (!root) return;

 const $ = id => root.querySelector("#amn-" + id);
 const MODEL = "gemini-3-flash-preview";

 let current = null;
 let history = [];

 function escapeHTML(s) {
   return String(s ?? "").replace(/[&<>"']/g, c => ({
     "&":"&amp;","<":"&lt;",">":"&gt;",
     '"':"&quot;","'":"&#39;"
   })[c]);
 }

 function normalize(data) {
   return {
     summary: String(data.summary || "No summary provided."),
     tasks: Array.isArray(data.tasks) ? data.tasks.map(t => ({
       task: String(t.task || "Unnamed task"),
       owner: String(t.owner || "Not specified"),
       deadline: String(t.deadline || "Not specified"),
       priority: ["High","Medium","Low"].includes(t.priority)
         ? t.priority : "Medium",
       status: ["Completed","In Progress","Pending"].includes(t.status)
         ? t.status : "Pending"
     })) : [],
     decisions: Array.isArray(data.decisions)
       ? data.decisions.map(String) : []
   };
 }

 function priorityFor(text) {
   if (/\b(urgent|immediately|critical|asap|high priority)\b/i.test(text))
     return "High";
   if (/\b(important|soon|priority|deadline)\b/i.test(text))
     return "Medium";
   return "Low";
 }

 function offlineExtract(text) {
   const sentences = text.split(/[\n.!?]+/)
     .map(s => s.trim()).filter(Boolean);

   const keywords = /\b(will|must|should|need to|complete|prepare|submit|finish|develop|test|create|deliver|fix|review|send|build)\b/i;

   const tasks = sentences.filter(s => keywords.test(s)).map(s => {
     const ownerMatch = s.match(/^([A-Z][a-z]+)\s+/);
     const deadlineMatch = s.match(/\b(by\s+)?(Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday|tomorrow|today|next week|next month)\b/i);

     return {
       task: s,
       owner: ownerMatch ? ownerMatch[1] : "Not specified",
       deadline: deadlineMatch ? deadlineMatch[0] : "Not specified",
       priority: priorityFor(s),
       status: /\b(completed|finished|done)\b/i.test(s) ? "Completed" : "Pending"
     };
   });

   const decisions = sentences.filter(s =>
     /\b(decided|agreed|approved|decision)\b/i.test(s)
   );

   return normalize({
     summary: "Local offline analysis of the meeting notes. This is a basic rule-based summary.",
     tasks,
     decisions
   });
 }

 async function geminiExtract(text) {
   const KEY = $("api-key").value.trim();
   if (!KEY) throw new Error("Enter your Gemini API key to use Gemini mode.");
   const response = await fetch(
     `https://generativelanguage.googleapis.com/v1beta/models/${MODEL}:generateContent`,
     {
       method:"POST",
       headers:{
         "Content-Type":"application/json",
         "x-goog-api-key":KEY
       },
       body:JSON.stringify({
         contents:[{
           parts:[{text:`Analyze the meeting notes below.

Return ONLY valid JSON. No markdown fences or additional text.

Required JSON structure:
{
 "summary": "Short meeting summary",
 "tasks": [
  {
   "task": "Task description",
   "owner": "Person or Not specified",
   "deadline": "Deadline or Not specified",
   "priority": "High, Medium, or Low",
   "status": "Pending, In Progress, or Completed"
  }
 ],
 "decisions": ["Decision 1"]
}

Rules:
- Extract genuine action items only.
- Never invent names or deadlines.
- Use Not specified for missing information.
- Assign High priority only to explicitly urgent or critical work.
- Use Medium for normal important tasks and Low for routine tasks.
- Do not treat a future launch date as a task deadline unless stated as one.

MEETING NOTES:
${text}`}]
         }],
         generationConfig:{responseMimeType:"application/json"}
       })
     }
   );

   const data = await response.json();

   if (!response.ok)
     throw new Error(data.error?.message || `HTTP ${response.status}`);

   const raw = data.candidates?.[0]?.content?.parts
     ?.map(p => p.text || "").join("");

   if (!raw) throw new Error("Gemini returned no text.");

   return normalize(JSON.parse(
     raw.replace(/^```json\s*/i,"").replace(/\s*```$/,"")
   ));
 }

 function render(data) {
   const tasks = data.tasks;
   $("total").textContent = tasks.length;
   $("high").textContent = tasks.filter(t => t.priority === "High").length;
   $("pending").textContent = tasks.filter(t => t.status !== "Completed").length;
   $("done").textContent = tasks.filter(t => t.status === "Completed").length;

   $("tasks").innerHTML = tasks.length ? tasks.map(t => `
    <tr>
     <td>${escapeHTML(t.task)}</td>
     <td>${escapeHTML(t.owner)}</td>
     <td>${escapeHTML(t.deadline)}</td>
     <td><span class="tag ${t.priority.toLowerCase()}">${escapeHTML(t.priority)}</span></td>
     <td>${escapeHTML(t.status)}</td>
    </tr>
   `).join("") :
   '<tr><td colspan="5">No action items identified.</td></tr>';

   $("summary").textContent = data.summary;
   $("decisions").textContent = data.decisions.length
     ? data.decisions.map((d,i) => `${i+1}. ${d}`).join("\n")
     : "No specific decisions identified.";
 }

 function reportText(data) {
   return [
    "MEETING INTELLIGENCE REPORT",
    "===========================",
    "",
    "MEETING SUMMARY",
    data.summary,
    "",
    "ACTION ITEMS",
    ...data.tasks.map((t,i) =>
      `${i+1}. ${t.task} | Owner: ${t.owner} | Deadline: ${t.deadline} | Priority: ${t.priority} | Status: ${t.status}`
    ),
    "",
    "IMPORTANT DECISIONS",
    ...data.decisions.map((d,i) => `${i+1}. ${d}`)
   ].join("\n");
 }

 function download(name, content, type) {
   const blob = new Blob([content],{type});
   const url = URL.createObjectURL(blob);
   const a = document.createElement("a");
   a.href = url;
   a.download = name;
   a.click();
   setTimeout(() => URL.revokeObjectURL(url),1000);
 }

 function renderHistory() {
   $("history").innerHTML = history.length ? "" :
     "No previous analyses in this session.";

   history.forEach((item,i) => {
     const div = document.createElement("div");
     div.className = "history-item";

     const title = document.createElement("strong");
     title.textContent = item.title;

     const info = document.createElement("div");
     info.className = "muted";
     info.textContent = `${item.data.tasks.length} tasks · ${item.mode}`;

     const button = document.createElement("button");
     button.className = "secondary";
     button.textContent = "View";
     button.addEventListener("click",() => {
       current = item.data;
       render(current);
       $("status").textContent = "Previous analysis restored.";
     });

     div.append(title,button,info);
     $("history").appendChild(div);
   });
 }

 $("run").addEventListener("click",async () => {
   const text = $("notes").value.trim();
   if (!text) {
     $("status").textContent = "Please enter meeting notes.";
     return;
   }

   $("run").disabled = true;
   $("run").textContent = "ANALYZING...";
   $("status").textContent = "Processing meeting notes...";

   try {
     const selectedMode = $("mode").value;

     const data = selectedMode === "live"
       ? await geminiExtract(text)
       : offlineExtract(text);

     current = data;
     render(data);

     history.unshift({
       title: text.slice(0,55) || "Meeting",
       data,
       mode:selectedMode === "live" ? "Gemini" : "Offline"
     });

     renderHistory();
     $("status").textContent = "Analysis completed successfully.";

   } catch(e) {
     $("status").textContent = "Analysis error: " + e.message;
   } finally {
     $("run").disabled = false;
     $("run").textContent = "✦ ANALYZE MEETING NOTES";
   }
 });

 $("txt").addEventListener("click",() => {
   if(current) download("meeting_report.txt",reportText(current),"text/plain");
 });

 $("csv").addEventListener("click",() => {
   if(!current) return;

   const rows = [["Task","Owner","Deadline","Priority","Status"],
     ...current.tasks.map(t =>
       [t.task,t.owner,t.deadline,t.priority,t.status]
     )];

   const csv = rows.map(row =>
     row.map(v => '"' + String(v).replace(/"/g,'""') + '"').join(",")
   ).join("\n");

   download("meeting_tasks.csv",csv,"text/csv");
 });

 $("copy").addEventListener("click",async () => {
   if(!current) return;
   try {
     await navigator.clipboard.writeText(reportText(current));
     $("status").textContent = "Report copied.";
   } catch(e) {
     $("status").textContent = "Clipboard unavailable in this browser.";
   }
 });

 $("pdf").addEventListener("click",() => {
   if(!current) {
     $("status").textContent = "Generate a report first.";
     return;
   }
   window.print();
 });
})();
"""

display(Javascript(js))
